# 01 · Dati: download, merge, pulizia — solo cervello, solo umano, senza sangue

**Perimetro.** Restano solo proteine **umane** (UniProt revisionate), **espresse nel cervello** (tessuti cerebrali
HPA, neuroni o glia, o annotate come cerebrali in UniProt) e **non del sangue**. Vale per il training, per il test e
per le predizioni. Le etichette di localizzazione vengono da qualunque esperimento umano (anche linee cellulari):
misure di localizzazione su migliaia di proteine nel tessuto cerebrale umano non esistono, e i segnali di
smistamento degli organelli sono gli stessi in tutte le cellule. Le altre specie di DeepLoc restano fuori
(`ALTRE_SPECIE = False`); test reale: tenerle con peso ridotto vale circa +0,05 di AUPRC sul test umano, ma il
perimetro del progetto è umano.

**Classi da predire (14).** Nove compartimenti generali (Citoplasma, Nucleo, Extracellulare, Membrana plasmatica,
Mitocondrio, Reticolo endoplasmatico, Lisosoma/Vacuolo, Golgi, Perossisoma — lo schema di DeepLoc 2.0, così il
confronto è alla pari) più cinque **compartimenti del neurone**: Assone, Dendrite, Presinapsi, Postsinapsi, Mielina.
I compartimenti neuronali sono il valore aggiunto per i ricercatori sulle malattie neurologiche e sono, per natura,
le classi più rare: per bilanciarle il più possibile si raccolgono etichette da tutte le fonti neuronali disponibili
(UniProt, HPA, SynGO) e, come fonti aggiuntive facoltative, le banche dati del sinaptoma — **SynGO** (presinapsi /
postsinapsi curate) e **SynaptomeDB / Synaptome.db** (proteoma sinaptico, compartimenti pre/post). Si scaricano a
mano e si convertono in `data/esterni/fonti/<nome>.csv` (colonne `gene`, `classe`); il notebook le integra come le
altre fonti, con il canale di rumore che ne stima l'affidabilità. Lo squilibrio residuo tra classi è gestito nel
notebook 02 dai pesi dei positivi per classe e dalla selezione sulla **macro**-AUPRC (ogni classe pesa uguale).

**card_protein.** Per ogni proteina una scheda testuale (in inglese, la lingua dei modelli biomedici) con quello che
un esperto vuole sapere: nome, famiglia, funzione, attività, domini, partner, modifiche, processi e funzioni GO,
espressione nel cervello e nei tipi cellulari, malattie, complessi. **Senza nessuna informazione di localizzazione**:
il campo UniProt della localizzazione, i termini GO di componente cellulare e ogni parola che nomina un compartimento
vengono tolti, altrimenti il modello leggerebbe la risposta invece di impararla.

Il notebook costruisce quattro cose:
1. una tabella di **proteine** con la sequenza;
2. una tabella **lunga di osservazioni**: per ogni proteina, ogni fonte e ogni compartimento, la fonte dice
   1 (presente), 0 (assente) oppure niente (non misurato);
3. i **target ausiliari** (meccanismi di sorting e PTM) e il **contesto genico**: sono le caratteristiche che il vecchio
   notebook FFFFF ricavava da UniProt, HPA, MitoCarta e SynaptopathyDB, ora calcolate direttamente qui;
4. le **varianti** ClinVar e, se disponibili, quelle con localizzazione misurata sperimentalmente.

**Perché una tabella di osservazioni invece di un'etichetta unica.** Le fonti hanno affidabilità diverse e note:
DeepLoc (sperimentale, curata) è l'ancora; UniProt è incompleto; HPA Approved è meno solido di Enhanced; le mappe
neuronali di topo sono un'altra specie. Invece di fonderle a mano con soglie arbitrarie, il notebook 02 impara per
ogni fonte e classe **sensibilità** (quante localizzazioni vere riporta) e **specificità** (quanti falsi positivi).
Così si usano anche le etichette deboli senza farsi ingannare.

**Regola delle assenze.** Per ogni fonte, una proteina che la fonte riporta e un compartimento che la fonte sa
misurare ma non assegna a quella proteina = osservazione 0. Tutto il resto = non osservato. Esempio: HPA non misura
l'extracellulare, quindi per HPA quella classe resta sempre vuota.

**Fonti automatiche:** DeepLoc 2.0 (training + test HPA indipendente), UniProt umano, HPA subcellular,
HPA RNA tessuti e single-cell, OpenCell, MitoCarta 3.0, SynaptopathyDB, ClinVar.

**Cosa è rimasto di FFFFF e cosa no.** Restano: query UniProt cerebrale, filtro di espressione cerebrale HPA,
MitoCarta, SynaptopathyDB, ClinVar, le feature di PTM / NLS / multimero / aggregati / patologia cerebrale / ruolo
di trafficking / struttura PDB (ora dal campo UniProt, senza interrogare RCSB). Sono state tolte: le ~9000 colonne
GO (contengono la risposta), COMPARTMENTS (integra UniProt e GO, quindi circolare), Allen (sostituito dai dati HPA
di tessuto e single-cell), PDB via RCSB, le celle "PIR" e "SynGO" di FFFFF (erano query UniProt ripetute; il vero SynGO ora entra come fonte), synaptosome ed ExoCarta
(le pagine scaricate non contenevano dati).

**Fonti da scaricare a mano** (due file, entrambi pubblici e senza registrazione; se mancano il notebook va avanti senza):
- **MitoCarta 3.0**: broadinstitute.org/mitocarta → *Human MitoCarta3.0* (`Human.MitoCarta3.0.xls`). Va in `cache/` con il nome
  originale. Per leggere il `.xls` serve `pip install xlrd`. Si usa il foglio "B Human All Genes": i 19.247 geni valutati.
- **SynGO**: syngoportal.org → Download → *bulk download*. Lo zip (con `annotations`, `genes`, `ontologies`) va in
  `data/esterni/` così com'è, senza estrarlo.

**Fonti scaricate in automatico** oltre a quelle elencate sopra: GO umano con le annotazioni sperimentali
(`goa_human.gaf.gz`, EBI) e l'ontologia (`go-basic.obo`). Il file `.obo` aperto nel browser sembra un enorme file di testo:
è normale, è il formato dell'ontologia; qui lo scarica il codice.

**Facoltative** (vuote vanno bene): `data/esterni/fonti/<nome>.csv` (colonne `gene`, `classe`), `data/esterni/varianti/<nome>.csv`
(colonne `gene`, `variante`, `mislocalizzata`) e il proteoma AlphaFold estratto in `data/alphafold/` (i file
`AF-<accession>-F1-model_v*.cif.gz` vengono decompressi; altrimenti lo 02 scarica le strutture una per una).

### Domande da esperto sulle fonti aggiunte
- *Come si collegano i geni di SynGO alle proteine?* Non con la colonna `uniprot_id` di SynGO: quasi sempre è la proteina
  di ratto o di topo dell'esperimento originale. Si usa HGNC → Entrez (file `genes`) → GeneID di UniProt, poi il simbolo.
- *Quali termini SynGO diventano presinapsi o postsinapsi?* Quelli che discendono da *presynapse* (GO:0098793) o da
  *postsynapse* (GO:0098794) nella gerarchia del file `ontologies`, compresi ribosomi, proteasomi e RE presinaptici e
  postsinaptici (traduzione e degradazione locali). *Synapse*, *synaptic membrane* e fessura non dicono il lato.
- *Tutte le annotazioni SynGO valgono uguale?* No: microscopia e frazionamento mostrano dove sta la proteina (fonte
  `syngo`), elettrofisiologia e interazioni dove agisce (fonte `syngo_funzionale`). Le annotazioni ottenute solo in
  cellule non neuronali si scartano: lì non ci sono sinapsi. Sui file reali 463 geni su 1.461 (32%) sono annotati ai due
  lati della sinapsi: presinapsi e postsinapsi sono classi indipendenti.
- *Perché il foglio "B" di MitoCarta?* Elenca tutti i geni valutati: uno valutato e non in lista è un'assenza vera. Le 80
  proteine "mito-interacting" (enzimi agganciati alla membrana esterna come HK2) restano non osservate. Si collega per
  la colonna UniProt, poi per simbolo.
- *Perché GOA se c'è già UniProt?* Si tengono solo le parti indipendenti: componente cellulare con evidenza sperimentale
  (EXP, IDA, HDA, HTP), senza NOT e "colocalizes_with", escluse le annotazioni assegnate da HPA, UniProt e SynGO, che
  duplicherebbero fonti già presenti. Restano curatori dedicati alla neurodegenerazione (ARUK-UCL, ParkinsonsUK-UCL) e
  screening su larga scala, utili anche per assone, dendrite e mielina.
- *GOA dà solo positivi: si tratta come fonte "solo positivi"?* No: con soli positivi la specificità non è stimabile e il
  canale di rumore la porterebbe a zero. Si applica la regola delle assenze come per UniProt, anch'essa incompleta: lo 02
  stima una sensibilità bassa, così un'assenza pesa poco e una presenza molto.

Output: `data/proteine.parquet`, `data/osservazioni.parquet`, `data/grafo.parquet`, `data/clinvar_missense.parquet`,
`data/varianti_sperimentali.parquet`

In [ ]:
import io
import gzip
import re
import zipfile
from itertools import compress
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import cross_val_predict
from sklearn.svm import SVC
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CACHE, DATA, EXT = Path('cache'), Path('data'), Path('data/esterni')
for d in [CACHE, DATA, EXT / 'fonti', EXT / 'varianti']:
    d.mkdir(parents=True, exist_ok=True)

CLASSES = ['Cytoplasm', 'Nucleus', 'Extracellular', 'Cell membrane', 'Mitochondrion',
           'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus', 'Peroxisome']
NEURO = ['Assone', 'Dendrite', 'Presinapsi', 'Postsinapsi', 'Mielina']     # la mielina è la membrana degli oligodendrociti
ESCLUDI_SANGUE = True        # proteine del sangue: fuori dalle predizioni e dalle valutazioni sul cervello
BRAIN_NTPM, NEURON_NCPM = 1, 5        # 1 nTPM = "rilevata" nel tessuto (soglia HPA); 5 nCPM per i compartimenti neuronali
ALTRE_SPECIE = False                  # True = tiene anche le proteine non umane di DeepLoc nel training

session = requests.Session()
session.headers['User-Agent'] = 'brain-misloc/3.0 (academic research; UNISA)'
session.mount('https://', HTTPAdapter(max_retries=Retry(total=6, backoff_factor=2,
                                                        status_forcelist=[429, 500, 502, 503, 504])))

In [ ]:
UNIPROT_FIELDS = ('accession,gene_names,protein_name,sequence,cc_subcellular_location,xref_geneid,'
                  'ft_signal,ft_transit,ft_transmem,ft_lipid,keyword,cc_ptm,cc_function,cc_disease,cc_subunit,'
                  'ft_region,ft_motif,xref_pdb,cc_tissue_specificity,cc_domain,ft_domain,protein_families,cc_pathway,'
                  'go_p,go_f,cc_catalytic_activity,annotation_score')
BRAIN_QUERY = ('reviewed:true AND organism_id:9606 AND (cc_tissue_specificity:brain OR cc_tissue_specificity:neuron '
               'OR cc_tissue_specificity:neuro* OR cc_tissue_specificity:cerebr* OR cc_tissue_specificity:hippocampus '
               'OR cc_tissue_specificity:cortex OR cc_tissue_specificity:astrocyte OR cc_tissue_specificity:microglia '
               'OR cc_tissue_specificity:glia OR cc_tissue_specificity:"nervous system" OR keyword:KW-0165 '
               'OR keyword:KW-0523 OR keyword:KW-0524 OR go:0007268 OR go:0030182 OR go:0007411 OR go:0045202 '
               'OR go:0043005 OR go:0098794 OR go:0014069)')
STREAM = 'https://rest.uniprot.org/uniprotkb/stream'
DTU = 'https://services.healthtech.dtu.dk/services/DeepLoc-2.0/data/'
HPA = 'https://www.proteinatlas.org/download/tsv/'
ORGIP = 'https://raw.githubusercontent.com/czbiohub-sf/Organelle_IP_analyses_and_figures/main/'   # Hein et al. 2024, licenza BSD

sources = pd.DataFrame([
    ('deeploc_train.csv', DTU + 'Swissprot_Train_Validation_dataset.csv', None),
    ('deeploc_hpa_test.csv', DTU + 'hpa_testset.csv', None),
    ('uniprot_human_v5.tsv', STREAM, {'query': 'reviewed:true AND organism_id:9606 AND fragment:false',
                                      'fields': UNIPROT_FIELDS, 'format': 'tsv'}),
    ('uniprot_brain_ids.tsv', STREAM, {'query': BRAIN_QUERY, 'fields': 'accession', 'format': 'tsv'}),
    ('hpa_subcellular.zip', HPA + 'subcellular_location.tsv.zip', None),
    ('hpa_rna_tissue.zip', HPA + 'rna_tissue_consensus.tsv.zip', None),
    ('hpa_rna_single_cell.zip', HPA + 'rna_single_cell_type.tsv.zip', None),
    ('synaptopathydb_genes.csv', 'https://www.synaptopathydb.org/api/genes', None),
    ('opencell_localization.csv', 'https://opencell.sf.czbiohub.org/data/datasets/opencell-localization-annotations.csv', None),
    ('hpa_proteinatlas.zip', 'https://www.proteinatlas.org/download/proteinatlas.tsv.zip', None),
    ('hein2024_noc.csv', ORGIP + 'notebooks/enrichment/output/enrichment_and_volcano_tables/2023-10-21-imp5-for-figures_enrichment_table_NOC_prop.csv', None),
    ('itzhak2016.xlsx', ORGIP + 'data/external/elife-16950-supp9-v3.xlsx', None),
    ('corum4.tsv', ORGIP + 'data/external/Corum.4.0.tsv', None),
    ('lopit_hyper.csv', ORGIP + 'data/external/hyperLOPITU2OS2018.csv', None),
    ('lopit_dc.csv', ORGIP + 'data/external/lopitdcU2OS2018.csv', None),
    ('lopit_2024.csv', ORGIP + 'data/external/dLOPIT2024-DMSO.csv', None),
    ('marcatori_organelli.csv', ORGIP + 'data/external/curated_ground_truth_v9.0.csv', None),
    ('bioid2021_nmf.xlsx', ORGIP + 'data/external/BioID2021_NMF.xlsx', None),
    ('opencell_interactions.csv', ORGIP + 'data/external/opencell-protein-interactions.csv', None),
    ('opencell_abundance.csv', ORGIP + 'data/external/opencell-protein-abundance.csv', None),
    # MitoCarta 3.0 e SynGO si scaricano a mano (il Broad blocca i download automatici; SynGO dà uno zip): vedi l'intestazione
    ('clinvar_variant_summary.txt.gz', 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz', None),
    ('goa_human.gaf.gz', 'https://ftp.ebi.ac.uk/pub/databases/GO/goa/HUMAN/goa_human.gaf.gz', None),   # GO umano, annotazioni
    ('go-basic.obo', 'https://purl.obolibrary.org/obo/go/go-basic.obo', None),                        # GO, nomi dei termini
    # varianti con localizzazione misurata: lo screening di Lacoste et al. 2024 nella versione curata da Wefers et al. 2026
    ('lacoste_wefers2026.csv', 'https://raw.githubusercontent.com/CellProfiling/seq2loc_benchmark/main/datasets/final/lacoste_metadata.csv', None),
], columns=['file', 'url', 'params'])

# scarico solo i file mancanti; il .part evita file troncati in cache se il download si interrompe
missing = sources[[not (CACHE / f).exists() for f in sources['file']]]
for f, url, params in missing.itertuples(index=False):
    with session.get(url, params=params, stream=True, timeout=3600) as r, open(CACHE / f'{f}.part', 'wb') as out:
        r.raise_for_status()
        for chunk in r.iter_content(1 << 20):
            out.write(chunk)
    (CACHE / f'{f}.part').replace(CACHE / f)
    print('scaricato', f)

## Proteine: DeepLoc (training con partizioni di omologia), test HPA, UniProt umano

In [ ]:
dl = pd.read_csv(CACHE / 'deeploc_train.csv', index_col=0).rename(
    columns={'ACC': 'uniprot_id', 'Sequence': 'sequence', 'Partition': 'partition'})
dl['source'] = 'deeploc'
hpa_test = pd.read_csv(CACHE / 'deeploc_hpa_test.csv').rename(columns={'sid': 'id_test', 'fasta': 'sequence'})
hpa_test['source'], hpa_test['partition'] = 'hpa_test', -1

up = pd.read_csv(CACHE / 'uniprot_human_v5.tsv', sep='\t', dtype=str).rename(columns={
    'Entry': 'uniprot_id', 'Gene Names': 'gene_names', 'Protein names': 'protein_name', 'Sequence': 'sequence',
    'Subcellular location [CC]': 'scl', 'GeneID': 'geneid', 'Signal peptide': 'sig', 'Transit peptide': 'transit',
    'Transmembrane': 'tm', 'Lipidation': 'lipid', 'Keywords': 'keywords', 'Post-translational modification': 'ptm',
    'Function [CC]': 'function', 'Involvement in disease': 'disease', 'Subunit structure': 'subunit', 'Region': 'region',
    'Motif': 'motif', 'PDB': 'pdb', 'Tissue specificity': 'tissue', 'Domain [CC]': 'domain_cc', 'Domain [FT]': 'domain_ft',
    'Protein families': 'family', 'Pathway': 'pathway', 'Gene Ontology (biological process)': 'go_bp',
    'Gene Ontology (molecular function)': 'go_mf', 'Catalytic activity': 'catalytic', 'Annotation': 'annotation'})
CARD_FIELDS = ['tissue', 'domain_cc', 'domain_ft', 'family', 'pathway', 'go_bp', 'go_mf', 'catalytic', 'annotation']
up = up.reindex(columns=list(dict.fromkeys(list(up.columns) + CARD_FIELDS)))      # un campo assente resta vuoto
up['gene'] = up['gene_names'].fillna('').str.split().str[0].str.upper()   # in UniProt i nomi sono separati da spazi
brain_ids = set(pd.read_csv(CACHE / 'uniprot_brain_ids.tsv', sep='\t')['Entry'])

# simbolo genico -> UniProt: simbolo ufficiale, poi alias solo se non ambigui (serve per HPA, MitoCarta, fonti esterne)
tokens = up[['uniprot_id', 'gene_names']].assign(sym=up['gene_names'].fillna('').str.upper().str.split()).explode('sym')
alias_n = tokens.groupby('sym')['uniprot_id'].transform('nunique')
sym2id = pd.concat([up.set_index('gene')['uniprot_id'], tokens[alias_n == 1].set_index('sym')['uniprot_id']])
sym2id = sym2id[~sym2id.index.duplicated() & (sym2id.index != '')]

# il test HPA di DeepLoc 2.0 usa sequenze Ensembl: solo una parte coincide con la sequenza canonica di UniProt. HPA misura
# la localizzazione per gene (con un anticorpo), quindi dove la sequenza non coincide la proteina si collega per gene
# (ID Ensembl del gene, poi simbolo) e prende la sequenza canonica di UniProt, come tutte le altre proteine del progetto
with zipfile.ZipFile(CACHE / 'hpa_proteinatlas.zip') as z:
    atl = pd.read_csv(z.open(z.namelist()[0]), sep='\t', low_memory=False)
atl_ens = pd.concat([atl.filter(regex='(?i)^ensembl$').iloc[:, :1], atl.filter(regex='(?i)^uniprot$').iloc[:, :1],
                     pd.DataFrame(index=atl.index, columns=['_'])], axis=1)
atl_ens.columns = ['ens', 'acc'][:atl_ens.shape[1] - 1] + ['_']
atl_ens = atl_ens.reindex(columns=['ens', 'acc']).astype(str)
atl_ens['acc'] = atl_ens['acc'].str.split(r'[,;\s]+').str[0]
atl_ens = atl_ens[atl_ens['acc'].isin(set(up['uniprot_id']))].drop_duplicates('ens')
ens2up = pd.Series(atl_ens['acc'].to_numpy(), index=atl_ens['ens'].to_numpy())
per_seq = hpa_test['sequence'].str.upper().map(up.drop_duplicates('sequence').set_index('sequence')['uniprot_id'])
per_gene = (hpa_test['id_test'].astype(str).str.extract(r'(ENSG\d+)', expand=False).map(ens2up)
            .fillna(hpa_test['id_test'].astype(str).str.upper().map(sym2id)))
hpa_test['uniprot_id'] = per_seq.fillna(per_gene)
hpa_test['sequence'] = hpa_test['uniprot_id'].map(up.drop_duplicates('uniprot_id').set_index('uniprot_id')['sequence']).fillna(hpa_test['sequence'])
print(f"Test HPA di DeepLoc 2.0: {len(hpa_test):,} proteine | collegate a UniProt per sequenza identica {per_seq.notna().sum():,}, "
      f"per gene {(per_seq.isna() & per_gene.notna()).sum():,}, non collegate {hpa_test['uniprot_id'].isna().sum():,}")
hpa_test = pd.concat([hpa_test.dropna(subset=['uniprot_id']).drop_duplicates('uniprot_id'), hpa_test[hpa_test['uniprot_id'].isna()]],
                     ignore_index=True)
# DeepLoc usa accession di UniProt: quelle umane cambiate nel tempo si ritrovano per sequenza identica
seq2id = up.drop_duplicates('sequence').set_index('sequence')['uniprot_id']
dl['uniprot_id'] = dl['uniprot_id'].where(dl['uniprot_id'].isin(set(up['uniprot_id'])), dl['sequence'].map(seq2id).fillna(dl['uniprot_id']))
dl['is_human_dl'] = dl['uniprot_id'].isin(set(up['uniprot_id']))
print(f"DeepLoc: {len(dl):,} proteine, umane {dl['is_human_dl'].sum():,}")

## Osservazioni da UniProt
Ogni frase del campo di localizzazione ha la forma `Termine {ECO:...}`. Il codice ECO dice l'evidenza:
sperimentale (269, 314, 7005, 353) o inferenza del curatore (305) → fonte `uniprot_forte`; per similarità o
predizione → fonte `uniprot_debole`. Le due fonti avranno sensibilità e specificità diverse.

In [ ]:
scl = up.set_index('uniprot_id')['scl'].fillna('')
scl = scl.str.replace(r'SUBCELLULAR LOCATION: \[Isoform (?!1\])[^\]]*\]:.*?(?=SUBCELLULAR LOCATION:|$)', '', regex=True)
scl = scl.str.replace(r'Note=.*?(?=SUBCELLULAR LOCATION:|$)', '', regex=True)
scl = scl.str.replace(r'SUBCELLULAR LOCATION:|\[Isoform 1\]:', '.', regex=True)
terms = scl.str.extractall(r'([A-Za-z][^.;{}]*?)\s*\{([^}]*)\}').rename(columns={0: 'term', 1: 'eco'}).reset_index()
term = terms['term'].str.strip().str.lower()

# termine -> organello: le regole vanno dalla più generica alla più specifica, l'ultima che fa match vince
RULES = [
    ('Cytoplasm', r'cytoplasm|cytosol|cytoskeleton|centrosome|centriole|microtubule|perikaryon|actin|spindle|'
                  r'p-body|stress granule|myofibril|sarcomere|z line'),
    ('Nucleus', r'\bnucleus\b|nucleolus|nucleoplasm|chromosome|centromere|kinetochore|telomere|pml body|cajal|'
                r'nuclear (?:body|speckle|pore|envelope|membrane|matrix)'),
    ('Cell membrane', r'cell membrane|plasma membrane|cell surface|junction|sarcolemma|t-tubule|synaptic cell membrane'),
    ('Extracellular', r'^secreted|extracellular space|extracellular matrix|basement membrane'),
    ('Endoplasmic reticulum', r'endoplasmic reticulum|sarcoplasmic reticulum|microsome'),
    ('Golgi apparatus', r'golgi'),
    ('Mitochondrion', r'mitochondri'),
    ('Lysosome/Vacuole', r'lysosom|vacuol|melanosom|autophagosom'),
    ('Peroxisome', r'peroxisom'),
]
terms['classe'] = None
for cls, pattern in RULES:
    terms.loc[term.str.contains(pattern), 'classe'] = cls
# topologia di membrana, vescicole generiche, esosomi: non sono organelli ("Cytoplasmic vesicle" NON è citoplasma)
terms.loc[term.str.contains(r'single-pass|multi-pass|peripheral membrane|lipid-anchor|gpi-anchor|\bside$|^membrane$|'
                            r'vesicle|endosom|exosome|granule membrane'), 'classe'] = None

# termine -> compartimento neuronale (stesse frasi, regole proprie: qui le vescicole sinaptiche contano)
neuro_term = pd.Series(None, index=terms.index, dtype=object)
for cls, pattern in [('Assone', r'\baxon'), ('Dendrite', r'dendrit|spine'),
                     ('Presinapsi', r'presynap|synaptic vesicle|active zone|synaptic bouton'),
                     ('Postsinapsi', r'postsynap|postsynaptic density'), ('Mielina', r'myelin')]:
    neuro_term[term.str.contains(pattern)] = cls
terms = pd.concat([terms, terms.assign(classe=neuro_term)], ignore_index=True).dropna(subset=['classe'])
strong = terms['eco'].str.contains(r'ECO:0000269|ECO:0000314|ECO:0007005|ECO:0000353|ECO:0000305')
obs_uniprot = terms.assign(fonte=np.where(strong, 'uniprot_forte', 'uniprot_debole'))[['uniprot_id', 'fonte', 'classe']]

## Osservazioni da HPA, MitoCarta e fonti esterne

In [ ]:
with zipfile.ZipFile(CACHE / 'hpa_subcellular.zip') as z:
    hpa = pd.read_csv(z.open(z.namelist()[0]), sep='\t')
HPA_MAP = {'Nucleoplasm': 'Nucleus', 'Nuclear speckles': 'Nucleus', 'Nuclear membrane': 'Nucleus', 'Nucleoli': 'Nucleus',
           'Nucleoli fibrillar center': 'Nucleus', 'Nucleoli rim': 'Nucleus', 'Nuclear bodies': 'Nucleus',
           'Kinetochore': 'Nucleus', 'Mitotic chromosome': 'Nucleus', 'Cytosol': 'Cytoplasm', 'Centrosome': 'Cytoplasm',
           'Centriolar satellite': 'Cytoplasm', 'Microtubules': 'Cytoplasm', 'Intermediate filaments': 'Cytoplasm',
           'Actin filaments': 'Cytoplasm', 'Aggresome': 'Cytoplasm', 'Cytoplasmic bodies': 'Cytoplasm',
           'Mitotic spindle': 'Cytoplasm', 'Plasma membrane': 'Cell membrane', 'Cell Junctions': 'Cell membrane',
           'Focal adhesion sites': 'Cell membrane', 'Endoplasmic reticulum': 'Endoplasmic reticulum',
           'Golgi apparatus': 'Golgi apparatus', 'Mitochondria': 'Mitochondrion', 'Lysosomes': 'Lysosome/Vacuole',
           'Peroxisomes': 'Peroxisome'}
hpa_long = hpa.melt(id_vars='Gene name', value_vars=['Enhanced', 'Supported', 'Approved'], var_name='rel', value_name='loc').dropna()
hpa_long = hpa_long.assign(loc=hpa_long['loc'].str.split(';')).explode('loc')
hpa_long['classe'] = hpa_long['loc'].str.strip().map(HPA_MAP)
hpa_long['fonte'] = np.where(hpa_long['rel'] == 'Approved', 'hpa_approved', 'hpa_alta')
hpa_long['uniprot_id'] = hpa_long['Gene name'].str.upper().map(sym2id)
# una proteina HPA "Approved" fa parte della fonte anche se la localizzazione è un organello non mappato (es. Vesicles):
# per questo tengo le righe con classe vuota, servono a dire che la proteina è stata misurata
obs_hpa = hpa_long.dropna(subset=['uniprot_id'])[['uniprot_id', 'fonte', 'classe']]

# MitoCarta 3.0 (a mano, in cache/ o data/esterni/): foglio B = tutti i geni valutati; se c'è solo il foglio A, l'universo
# sono tutte le proteine umane. I "mito-interacting" non sono né presenti né assenti
up_ids = set(up['uniprot_id'])
# i file scaricati a mano si cercano in più cartelle: la cartella del notebook, data/, cache/, data/esterni/ e le sue sottocartelle
CARTELLE_MANO = [Path('.'), DATA, CACHE, EXT, EXT / 'mitocarta', EXT / 'syngo']
# un file danneggiato (per esempio un download o un caricamento interrotto) non deve fermare il notebook né passare in
# silenzio: con xlrd si blocca, con altri lettori di .xls si legge solo una parte del foglio B
fogli = []
for f in [f for d in CARTELLE_MANO for f in sorted(d.glob('*[Mm]ito[Cc]arta*.xls*'))]:
    try:
        fogli.append(pd.read_excel(f, sheet_name=None))
    except Exception as err:
        print(f'>>> ATTENZIONE: {f} non si legge ({type(err).__name__}): il file è danneggiato, va riscaricato da broadinstitute.org/mitocarta')
n_B = max([len(x.get('B Human All Genes', [])) for x in fogli] + [0])
print(['', f'>>> ATTENZIONE: il foglio B di MitoCarta ha {n_B:,} geni invece di 19.247: il file è incompleto, va riscaricato'][int(0 < n_B < 19000)])
VUOTO_MC = pd.DataFrame(columns=['Symbol', 'UniProt', 'MitoCarta3.0_List'])
mc = pd.concat([x.get('B Human All Genes', x.get('A Human MitoCarta3.0', VUOTO_MC)) for x in fogli] + [VUOTO_MC], ignore_index=True)
mc = mc.reindex(columns=['Symbol', 'UniProt', 'MitoCarta3.0_List'])
mc['MitoCarta3.0_List'] = mc['MitoCarta3.0_List'].fillna('MitoCarta3.0').astype(str)
mc['uniprot_id'] = mc['UniProt'].astype(str).str.split(r'[|;,\s]').str[0]
mc['uniprot_id'] = mc['uniprot_id'].where(mc['uniprot_id'].isin(up_ids), mc['Symbol'].astype(str).str.upper().map(sym2id))
mc = mc.dropna(subset=['uniprot_id']).drop_duplicates('uniprot_id')
HAS_B = int(any('B Human All Genes' in x for x in fogli))
stato_mc = up['uniprot_id'].map(mc.set_index('uniprot_id')['MitoCarta3.0_List']).fillna('0').to_numpy()
valutata = np.where(HAS_B, up['uniprot_id'].isin(set(mc['uniprot_id'])), True) & (stato_mc != 'mito-interacting') & (len(mc) > 0)
obs_mito = pd.DataFrame({'uniprot_id': up['uniprot_id'].to_numpy(), 'fonte': 'mitocarta', 'classe': 'Mitochondrion',
                         'obs': (stato_mc == 'MitoCarta3.0').astype(float)})[valutata]
print(f"MitoCarta: {int(obs_mito['obs'].sum()):,} mitocondriali su {len(obs_mito):,} proteine valutate (foglio B: {['no', 'sì'][HAS_B]})")
# senza il file MitoCarta il mitocondrio perde il suo inventario di riferimento: meglio saperlo subito
print(['', '>>> ATTENZIONE: MitoCarta non caricato. Metti human.mitocarta3.0.xls (integro, foglio B) in data/esterni/ e rilancia il notebook. '
            'Per leggere un .xls serve il pacchetto xlrd (pip install xlrd). Cartelle controllate: '
            + ', '.join(str(d.resolve()) for d in CARTELLE_MANO)][int(len(fogli) == 0)])

# OpenCell (tagging endogeno in HEK293T): gradi 3 e 2 = localizzazione netta; il grado 1 (debole) non conta
# né come presente né come assente
OC_MAP = {'cytoplasmic': 'Cytoplasm', 'cytoskeleton': 'Cytoplasm', 'centrosome': 'Cytoplasm',
          'nucleoplasm': 'Nucleus', 'nucleolus_gc': 'Nucleus', 'nucleolus_fc_dfc': 'Nucleus', 'nuclear_punctae': 'Nucleus',
          'nuclear_membrane': 'Nucleus', 'chromatin': 'Nucleus', 'membrane': 'Cell membrane', 'cell_contact': 'Cell membrane',
          'focal_adhesions': 'Cell membrane', 'er': 'Endoplasmic reticulum', 'golgi': 'Golgi apparatus',
          'mitochondria': 'Mitochondrion'}                     # 'vesicles' e 'big_aggregates' non sono classi DeepLoc
oc = pd.read_csv(CACHE / 'opencell_localization.csv')
oc = oc.melt(id_vars='target_name', value_vars=['annotations_grade_3', 'annotations_grade_2', 'annotations_grade_1'],
             var_name='grade', value_name='loc').dropna()
oc = oc.assign(loc=oc['loc'].str.split(r'[;,]')).explode('loc')
oc['classe'] = oc['loc'].str.strip().map(OC_MAP)
oc['uniprot_id'] = oc['target_name'].str.upper().map(sym2id)
oc['fonte'] = 'opencell'
obs_oc = oc[oc['grade'] != 'annotations_grade_1'].dropna(subset=['uniprot_id'])[['uniprot_id', 'fonte', 'classe']]
oc_weak = oc[oc['grade'] == 'annotations_grade_1'].dropna(subset=['uniprot_id', 'classe'])[['uniprot_id', 'fonte', 'classe']]

# SynGO (a mano): lo zip del bulk download in data/esterni/ letto direttamente, oppure i 3 file estratti in data/esterni/syngo/
zips = [zipfile.ZipFile(f) for d in CARTELLE_MANO for f in sorted(d.glob('*[Ss]yn[Gg][Oo]*.zip'))]
SG = {k: pd.concat([pd.read_excel(z.open(n)) for z in zips
                    for n in compress(z.namelist(), [bool(re.search(rf'^(?!__MACOSX).*{k}\.xlsx$', n)) for n in z.namelist()])] +
                   [pd.read_excel(f) for d in CARTELLE_MANO for f in sorted(d.glob(f'*{k}.xlsx'))] + [pd.DataFrame()], ignore_index=True)
      for k in ['annotations', 'genes', 'ontologies']}
SG = {k: v.drop_duplicates() for k, v in SG.items()}
ann = SG['annotations'].reindex(columns=['hgnc_id', 'hgnc_symbol', 'uniprot_id', 'go_id', 'go_domain', 'evidence_biological_system',
                                         'evidence_experiment_assay']).astype(str)
ont = SG['ontologies'].reindex(columns=['id', 'parent_id']).drop_duplicates('id')
# lato della sinapsi: si risale la gerarchia di SynGO fino a presynapse o postsynapse
LATI = {'GO:0098793': 'Presinapsi', 'GO:0098794': 'Postsinapsi'}
padre = ont.dropna(subset=['parent_id']).drop_duplicates('id').set_index('id')['parent_id']
anc = pd.Series(ont['id'].to_numpy(), index=ont['id'].to_numpy())
lato = anc.map(LATI)
for _ in range(10):
    anc = anc.map(padre).fillna(anc)
    lato = lato.fillna(anc.map(LATI))
# HGNC → Entrez (file genes) → GeneID di UniProt; poi il simbolo; poi l'accession, se è umana
entrez = up.assign(eid=up['geneid'].fillna('').str.strip(';').str.split(';')).explode('eid')
entrez2id = entrez[entrez['eid'] != ''].drop_duplicates('eid').set_index('eid')['uniprot_id']
hgnc2eid = (SG['genes'].reindex(columns=['hgnc_id', 'entrez_id']).dropna().astype({'entrez_id': int}).astype(str)
            .drop_duplicates('hgnc_id').set_index('hgnc_id')['entrez_id'])
ann['classe'] = ann['go_id'].map(lato)
ann['uid'] = (ann['hgnc_id'].map(hgnc2eid).map(entrez2id).fillna(ann['hgnc_symbol'].str.upper().map(sym2id))
              .fillna(ann['uniprot_id'].where(ann['uniprot_id'].isin(up_ids))))
neuronale = ann['evidence_biological_system'].str.contains('intacttissue|cultneuron')       # niente sinapsi in cellule non neuronali
localizza = ann['evidence_experiment_assay'].str.contains(r'mic:|frac:')                     # microscopia o frazionamento
ann = ann[(ann['go_domain'] == 'CC') & neuronale].dropna(subset=['uid', 'classe'])
obs_syngo = pd.DataFrame({'uniprot_id': ann['uid'], 'fonte': np.where(localizza[ann.index], 'syngo', 'syngo_funzionale'), 'classe': ann['classe']})
print(f'OpenCell: {obs_oc["uniprot_id"].nunique():,} proteine | SynGO: {obs_syngo["uniprot_id"].nunique():,} proteine')
# senza SynGO presinapsi e postsinapsi hanno solo qualche centinaio di positivi da UniProt: le classi sinaptiche restano deboli
print(['', '>>> ATTENZIONE: SynGO non trovato. Metti lo zip del bulk download (o i 3 file syngo_annotations/genes/ontologies.xlsx) '
            'in data/esterni/ e rilancia il notebook. Cartelle controllate: ' + ', '.join(str(d.resolve()) for d in CARTELLE_MANO)][int(len(obs_syngo) == 0)])

# GO umano: componente cellulare con evidenza sperimentale, senza le annotazioni di HPA, UniProt e SynGO (già fonti a sé)
go_name = pd.Series(dict(re.findall(r'\[Term\]\nid: (GO:\d{7})\nname: ([^\n]+)\nnamespace: cellular_component',
                                    (CACHE / 'go-basic.obo').read_text(errors='ignore'))), dtype=str)
gaf = pd.read_csv(CACHE / 'goa_human.gaf.gz', sep='\t', comment='!', header=None, usecols=[1, 3, 4, 6, 8, 14], dtype=str)
gaf = gaf[(gaf[8] == 'C') & gaf[6].isin(['EXP', 'IDA', 'HDA', 'HTP']) & ~gaf[3].fillna('').str.contains('NOT|colocalizes')
          & ~gaf[14].isin(['HPA', 'UniProt', 'SynGO', 'SynGO-UCL'])]
termine = gaf[4].map(go_name).fillna('').astype(str).str.lower()
GO_RULES = [('Cytoplasm', r'cytoplasm|cytosol|cytoskeleton|centrosome|centriole|microtubule|actin|spindle|p-body|stress granule|ribosome'),
            ('Nucleus', r'\bnucle(?:us|olus|oplasm)|nuclear (?:body|speck|pore|envelope|membrane|matrix|lamina)|chromosome|chromatin|'
                        r'kinetochore|centromer|telomer|pml body|cajal|spliceosom'),
            ('Cell membrane', r'plasma membrane|cell surface|cell-cell junction|adherens junction|tight junction|focal adhesion|cell junction'),
            ('Extracellular', r'extracellular (?:space|region|matrix)|basement membrane'),
            ('Endoplasmic reticulum', r'endoplasmic reticulum'), ('Golgi apparatus', r'golgi'), ('Mitochondrion', r'mitochondri'),
            ('Lysosome/Vacuole', r'lysosom|vacuol'), ('Peroxisome', r'peroxisom')]
cls_go = pd.Series(None, index=gaf.index, dtype=object)
for cls, pat in GO_RULES:
    cls_go[termine.str.contains(pat)] = cls                                      # le regole più specifiche vengono dopo
cls_go[termine.str.contains(r'vesicle|endosom|exosome|^membrane$|granule membrane')] = None
neu_go = pd.Series(None, index=gaf.index, dtype=object)
for cls, pat in [('Assone', r'\baxon|node of ranvier'), ('Dendrite', r'dendrit|spine'), ('Presinapsi', r'presynap|synaptic vesicle|active zone|bouton'),
                 ('Postsinapsi', r'postsynap'), ('Mielina', r'myelin')]:
    neu_go[termine.str.contains(pat)] = cls
obs_go = pd.DataFrame({'uniprot_id': pd.concat([gaf[1], gaf[1]]).to_numpy(), 'fonte': 'goa_exp',
                       'classe': pd.concat([cls_go, neu_go]).to_numpy()}).dropna()
obs_go = obs_go[obs_go['uniprot_id'].isin(up_ids)]
print(f"GO sperimentale: {obs_go['uniprot_id'].nunique():,} proteine, {len(obs_go):,} annotazioni")

# fonti esterne in formato lungo (gene, classe); i simboli di topo maiuscoli coincidono quasi sempre con quelli umani
ext_files = sorted((EXT / 'fonti').glob('*.csv'))
ext = pd.concat([pd.read_csv(f).assign(fonte=f.stem) for f in ext_files] +
                [pd.DataFrame(columns=['gene', 'classe', 'fonte'])], ignore_index=True)
ext['uniprot_id'] = ext['gene'].astype(str).str.strip().str.upper().map(sym2id)
# nomi di classe in inglese o in italiano ricondotti alle 14 classi
SINONIMI = {'cytoplasm': 'Cytoplasm', 'cytosol': 'Cytoplasm', 'nucleus': 'Nucleus', 'nuclear': 'Nucleus', 'extracellular': 'Extracellular',
            'secreted': 'Extracellular', 'cell membrane': 'Cell membrane', 'plasma membrane': 'Cell membrane', 'pm': 'Cell membrane',
            'mitochondrion': 'Mitochondrion', 'mitochondria': 'Mitochondrion', 'er': 'Endoplasmic reticulum',
            'endoplasmic reticulum': 'Endoplasmic reticulum', 'lysosome': 'Lysosome/Vacuole', 'lysosome/vacuole': 'Lysosome/Vacuole',
            'golgi': 'Golgi apparatus', 'golgi apparatus': 'Golgi apparatus', 'peroxisome': 'Peroxisome', 'axon': 'Assone',
            'axonal': 'Assone', 'assone': 'Assone', 'dendrite': 'Dendrite', 'dendritic': 'Dendrite', 'somatodendritic': 'Dendrite',
            'presynapse': 'Presinapsi', 'presynaptic': 'Presinapsi', 'presinapsi': 'Presinapsi', 'postsynapse': 'Postsinapsi',
            'postsynaptic': 'Postsinapsi', 'postsinapsi': 'Postsinapsi', 'myelin': 'Mielina', 'mielina': 'Mielina'}
ext['classe'] = ext['classe'].astype(str).str.strip().str.lower().map(SINONIMI).fillna(ext['classe'].astype(str).str.strip())
obs_ext = ext.dropna(subset=['uniprot_id'])[['uniprot_id', 'fonte', 'classe']]
obs_ext = obs_ext[obs_ext['classe'].isin(CLASSES + NEURO)]
print('Fonti esterne trovate:', [f.stem for f in ext_files])

## Proteomica spaziale umana: LOPIT e BioID (fonti "misurate", senza laboratorio)
Tre esperimenti LOPIT misurano come ogni proteina si distribuisce lungo un gradiente di centrifugazione:
hyperLOPIT e LOPIT-DC (U2OS, 2018) e dLOPIT (2024). Il profilo di una proteina somiglia a quello dell'organello in
cui sta. Come negli articoli originali, per ogni esperimento si addestra un classificatore (SVM) sui **marcatori**
curati (proteine a localizzazione nota) e lo si applica a tutte le altre. Una proteina diventa un'osservazione solo
se la probabilità della classe più probabile è ≥ 0,7; i marcatori ricevono la predizione in cross-validation, non la
propria etichetta. Test su dati reali: accuratezza sui marcatori 0,88–0,94, concordanza con DeepLoc 0,75–0,78 sulle
proteine non marcatrici (una fonte "rumorosa ma buona", come HPA Approved: il canale di rumore ne stima l'affidabilità).

BioID (Go et al. 2021, "Human Cell Map"): marcatura di prossimità da 192 esche; la fattorizzazione NMF dà per ogni
proteina un profilo su 20 "ranghi", ognuno annotato con il termine GO più arricchito. Una proteina è assegnata alle
classi che raccolgono almeno il 35% del suo profilo (più classi sono possibili). Concordanza con DeepLoc sulle
proteine non marcatrici: 0,80. Attenzione ai nomi: "ribonucleoprotein granule" (granuli da stress) è citoplasma,
non nucleo, per questo il pattern del nucleo cerca "nucle" solo a inizio parola.

In [ ]:
MK_MAP = {'nucleus': 'Nucleus', 'nuclear_pore': 'Nucleus', 'cytosol': 'Cytoplasm', 'ribosome': 'Cytoplasm',
          'stress granule': 'Cytoplasm', 'centrosome': 'Cytoplasm', 'actin-binding_protein': 'Cytoplasm', 'p-body': 'Cytoplasm',
          'mitochondria': 'Mitochondrion', 'ER': 'Endoplasmic reticulum', 'ERGIC': 'Endoplasmic reticulum',
          'plasma membrane': 'Cell membrane', 'lysosome': 'Lysosome/Vacuole', 'Golgi': 'Golgi apparatus', 'peroxisome': 'Peroxisome'}
markers = pd.read_csv(CACHE / 'marcatori_organelli.csv')
markers = pd.DataFrame({'uniprot_id': markers['gene_name_canonical'].str.upper().map(sym2id),
                        'classe': markers['compartment'].map(MK_MAP)}).dropna().drop_duplicates('uniprot_id').set_index('uniprot_id')['classe']
obs_lopit = []
for name, f in [('lopit_hyper', 'lopit_hyper.csv'), ('lopit_dc', 'lopit_dc.csv'), ('lopit_2024', 'lopit_2024.csv')]:
    prof = pd.read_csv(CACHE / f, index_col=0).dropna()
    prof.index = prof.index.astype(str).str.split(';').str[0].str.replace(r'-\d+$', '', regex=True)
    prof = prof[~prof.index.duplicated()]
    lab = markers.reindex(prof.index)
    is_mk = lab.notna().to_numpy()
    svm = SVC(C=10, gamma='scale', probability=True, class_weight='balanced', random_state=0)
    prob = np.zeros((len(prof), lab[is_mk].nunique()))
    prob[is_mk] = cross_val_predict(svm, prof.values[is_mk], lab[is_mk], cv=5, method='predict_proba')
    prob[~is_mk] = svm.fit(prof.values[is_mk], lab[is_mk]).predict_proba(prof.values[~is_mk])
    call = pd.DataFrame({'uniprot_id': prof.index, 'classe': svm.classes_[prob.argmax(1)], 'p': prob.max(1)})
    print(f"{name}: {len(prof):,} proteine, {is_mk.sum():,} marcatori, accuratezza sui marcatori (CV) "
          f"{(call['classe'].to_numpy()[is_mk] == lab.to_numpy()[is_mk]).mean():.2f}, assegnate con p ≥ 0,7: {(call['p'] >= 0.7).sum():,}")
    obs_lopit.append(call[call['p'] >= 0.7].assign(fonte=name)[['uniprot_id', 'fonte', 'classe']])

nmf = pd.read_excel(CACHE / 'bioid2021_nmf.xlsx', sheet_name='A) rank profiles').set_index('gene')
rank_term = pd.read_excel(CACHE / 'bioid2021_nmf.xlsx', sheet_name='B) enriched terms').groupby('rank').head(1).set_index('rank')['term name'].str.lower()
rank_cls = pd.Series('nessuna', index=rank_term.index)
# i termini più specifici vengono dopo e sovrascrivono (es. "nuclear outer membrane-endoplasmic reticulum" → RE)
for pattern, cls in [(r'cytosol|cytoskeleton|centrosome|ribonucleoprotein', 'Cytoplasm'), (r'\bnucle', 'Nucleus'),
                     (r'mitochondri', 'Mitochondrion'), (r'endoplasmic', 'Endoplasmic reticulum'), (r'golgi', 'Golgi apparatus'),
                     (r'plasma membrane|cell periphery', 'Cell membrane'), (r'lysosom', 'Lysosome/Vacuole'), (r'peroxisom', 'Peroxisome')]:
    rank_cls[rank_term.str.contains(pattern)] = cls
w = nmf.set_axis(nmf.columns.astype(str).str.split().str[-1].astype(int), axis=1).T.groupby(rank_cls).sum().T.drop(columns='nessuna', errors='ignore')
w = w.div(nmf.sum(axis=1).replace(0, np.nan), axis=0)
bio = w.stack().rename('quota').reset_index().rename(columns={'level_1': 'classe'})
bio = bio[bio['quota'] >= 0.35]
obs_bioid = pd.DataFrame({'uniprot_id': bio['gene'].astype(str).str.upper().map(sym2id), 'fonte': 'bioid', 'classe': bio['classe']}).dropna()
print(f"BioID: ranghi → classi {rank_cls.to_dict()} | proteine assegnate: {obs_bioid['uniprot_id'].nunique():,}")

## Regola delle assenze → tabella lunga di osservazioni
Per ogni fonte: proteine riportate × classi che la fonte sa misurare. Presente = 1, altrimenti 0.
MitoCarta è un inventario dell'intero genoma: ogni proteina valutata (foglio B) e non in lista è un'osservazione 0.

In [ ]:
pos = pd.concat([obs_uniprot, obs_hpa, obs_oc, obs_syngo, obs_go, obs_ext, obs_bioid] + obs_lopit, ignore_index=True)
covered = pos[['uniprot_id', 'fonte']].drop_duplicates()
measured = pos.dropna(subset=['classe'])[['fonte', 'classe']].drop_duplicates()
grid = covered.merge(measured, on='fonte')
hits = pos.dropna(subset=['classe']).drop_duplicates().assign(obs=1.0)
obs = grid.merge(hits, on=['uniprot_id', 'fonte', 'classe'], how='left').fillna({'obs': 0.0})
# le localizzazioni OpenCell di grado 1 restano "non osservate" invece di diventare assenze
obs = obs.merge(oc_weak.drop_duplicates().assign(debole=1), on=['uniprot_id', 'fonte', 'classe'], how='left')
obs = obs[obs['debole'].isna() | (obs['obs'] == 1)].drop(columns='debole')
# DeepLoc è la fonte "oro": tutte e 9 le classi osservate per ogni proteina
obs_dl = dl.melt(id_vars='uniprot_id', value_vars=CLASSES, var_name='classe', value_name='obs').assign(fonte='deeploc')
obs = pd.concat([obs_dl, obs, obs_mito], ignore_index=True)[['uniprot_id', 'fonte', 'classe', 'obs']]
print(obs.groupby('fonte').agg(proteine=('uniprot_id', 'nunique'), osservazioni=('obs', 'size'), positivi=('obs', 'sum')))

## Target ausiliari (meccanismi) e contesto genico (le feature utili di FFFFF)
I target ausiliari sono proprietà della proteina che dipendono dalla sequenza e spiegano *perché* va in un
compartimento: peptide segnale (via secretoria), transit peptide (mitocondrio), eliche TM (membrane), ancore
lipidiche/GPI (membrana), NLS (nucleo), glicosilazione (passaggio ER-Golgi), tendenza ad aggregare. Il modello
li deve **predire** dalla sequenza: non sono mai input. Sul mutante, la loro variazione indica il meccanismo rotto.
Il contesto genico (multi-localizzazione, densità di varianti patogeniche, espressione cerebrale) entra solo
nel modello delle varianti.

In [ ]:
aux = pd.DataFrame({'uniprot_id': up['uniprot_id'],
                    'aux_peptide_segnale': up['sig'].notna().astype(float),
                    'aux_transit_peptide': up['transit'].notna().astype(float),
                    'aux_tm_1piu': up['tm'].fillna('').str.count('TRANSMEM').ge(1).astype(float),
                    'aux_tm_2piu': up['tm'].fillna('').str.count('TRANSMEM').ge(2).astype(float),
                    'aux_ancora_lipidica': (up['lipid'].notna() | up['keywords'].fillna('').str.contains('GPI-anchor|Lipoprotein')).astype(float),
                    'aux_glicoproteina': up['keywords'].fillna('').str.contains('Glycoprotein').astype(float)})

# Feature ereditate da FFFFF, ricavate dai testi UniProt di tutte le proteine umane (non solo quelle "cerebrali")
fkw = (up['function'].fillna('') + ' ' + up['subunit'].fillna('') + ' ' + up['keywords'].fillna('')).str.lower()
ptm = (up['ptm'].fillna('') + ' ' + up['keywords'].fillna('')).str.lower()
dis = up['disease'].fillna('').str.lower()
syn_genes = set(pd.read_csv(CACHE / 'synaptopathydb_genes.csv')['HumanName'].astype(str).str.upper())
aux['aux_nls'] = (up['region'].fillna('') + up['motif'].fillna('') + up['keywords'].fillna('')).str.contains('(?i)nuclear localization signal')
aux['aux_multimero'] = fkw.str.contains(r'\bhomodimer|\bheterodimer|\bdimer|\btrimer|\btetramer|\boligomer|\bmultimer')
aux['aux_fosforilazione'] = ptm.str.contains(r'phosphorylat|phosphoprotein')
aux['aux_ubiquitinazione'] = ptm.str.contains(r'ubiquitinat|ubiquitylat|ubl conjugation')
aux['aux_palmitoilazione'] = ptm.str.contains(r'palmitoylat|palmitate')
aux['aux_processamento'] = ptm.str.contains(r'proteolytic|cleav|propeptide|zymogen')
aux['aux_aggregati'] = (dis + ' ' + ptm + ' ' + fkw).str.contains(r'amyloid|lewy bod|neurofibrillary tangle|inclusion bod|protein inclusion|aggregat')
aux['aux_patologia_cerebrale'] = dis.str.contains(r'alzheimer|parkinson|epilep|encephalopath|dementia|neurodegenerat|huntington|'
                                                  r'ataxia|intellectual disability|amyotrophic|leukodystroph|microcephal')
aux['aux_sinaptopatia'] = up['gene'].isin(syn_genes)
aux[aux.columns[1:]] = aux[aux.columns[1:]].astype(float)

# contesto genico (solo per il modello delle varianti): quanto la proteina è multi-localizzata, se fa la spola
# nucleo-citoplasma, se ha strutture risolte, che ruolo ha nel trafficking (stessa logica di FFFFF)
hpa_cls = hpa_long.dropna(subset=['classe', 'uniprot_id']).groupby('uniprot_id')['classe']
up_cls = obs_uniprot[obs_uniprot['classe'].isin(CLASSES)].groupby('uniprot_id')['classe']
ROLES = [('sorting receptor', r'sorting receptor|clathrin|endocytosis|vesicle trafficking'),
         ('channel', r'\bchannel\b|\bpore\b|\btransporter\b'),
         ('enzyme', r'\bkinase\b|\bphosphatase\b|\bprotease\b|\bligase\b|\bhydrolase\b|\bsynthase\b|\breductase\b'),
         ('receptor', r'\breceptor\b'), ('structural', r'\bstructural\b|\bscaffold\b|\bcytoskeletal\b|\bfilament\b')]
ctx_genic = pd.DataFrame({'uniprot_id': up['uniprot_id'],
                          'ctx_n_localizzazioni_hpa': up['uniprot_id'].map(hpa_cls.nunique()),
                          'ctx_n_localizzazioni_uniprot': up['uniprot_id'].map(up_cls.nunique()),
                          'ctx_shuttle_hpa': up['uniprot_id'].map(hpa_cls.agg(set).map({'Nucleus', 'Cytoplasm'}.issubset)).astype(float),
                          'ctx_shuttle_uniprot': up['uniprot_id'].map(up_cls.agg(set).map({'Nucleus', 'Cytoplasm'}.issubset)).astype(float),
                          'ctx_struttura_pdb': up['pdb'].notna().astype(float),
                          'ctx_ruolo_trafficking': np.select([fkw.str.contains(p) for _, p in ROLES], np.arange(1, 6), 6)})

with zipfile.ZipFile(CACHE / 'hpa_rna_tissue.zip') as z:
    rna = pd.read_csv(z.open(z.namelist()[0]), sep='\t', usecols=['Gene name', 'Tissue', 'nTPM'])
BRAIN_TISSUES = ['cerebral cortex', 'cerebellum', 'hippocampal formation', 'hippocampus', 'basal ganglia', 'hypothalamus',
                 'amygdala', 'midbrain', 'pons and medulla', 'pons', 'medulla oblongata', 'spinal cord', 'white matter',
                 'thalamus', 'choroid plexus']
brain_ntpm = rna[rna['Tissue'].str.lower().isin(BRAIN_TISSUES)].groupby(rna['Gene name'].str.upper())['nTPM'].max()
# espressione per tipo cellulare: i compartimenti neuronali hanno senso solo per proteine espresse nei neuroni
with zipfile.ZipFile(CACHE / 'hpa_rna_single_cell.zip') as z:
    sc = pd.read_csv(z.open(z.namelist()[0]), sep='\t', usecols=['Gene name', 'Cell type', 'nCPM'])
sc_gene = sc['Gene name'].str.upper()
neuron_ncpm = sc[sc['Cell type'].str.contains('neuron', case=False)].groupby(sc_gene)['nCPM'].max()
glia_ncpm = sc[sc['Cell type'].str.contains('astrocyte|oligodendro|microglia', case=False)].groupby(sc_gene)['nCPM'].max()

ctx = up[['uniprot_id', 'gene', 'geneid', 'protein_name']].copy()
ctx['brain_ntpm'] = ctx['gene'].map(brain_ntpm)
ctx['neuron_ncpm'] = ctx['gene'].map(neuron_ncpm)
ctx['glia_ncpm'] = ctx['gene'].map(glia_ncpm)
ctx['is_brain'] = (ctx['uniprot_id'].isin(brain_ids) | (ctx['brain_ntpm'] >= BRAIN_NTPM) |
                  (ctx['neuron_ncpm'] >= BRAIN_NTPM) | (ctx['glia_ncpm'] >= BRAIN_NTPM)).astype(int)
ctx['is_neuronal'] = (ctx['neuron_ncpm'] >= NEURON_NCPM).astype(int)
oligo_ncpm = sc[sc['Cell type'].str.contains('oligodendrocyte', case=False)].groupby(sc_gene)['nCPM'].max()
ctx['oligo_ncpm'] = ctx['gene'].map(oligo_ncpm)
ctx['is_oligo'] = (ctx['oligo_ncpm'] >= NEURON_NCPM).astype(int)
# proteine umane: DeepLoc contiene tutti gli eucarioti; "umana" = sequenza identica a una proteina umana UniProt
ctx['is_human'] = 1

## Sangue: cosa non appartiene al contesto cerebrale
Nel tessuto cerebrale e nelle sue banche dati finiscono proteine che nel cervello non vengono prodotte né
localizzate dalle cellule nervose: proteine plasmatiche sintetizzate dal fegato (albumina, fattori della
coagulazione...), proteine dei globuli rossi e dei leucociti. HPA classifica ogni proteina secreta per
destinazione ("Secreted to blood", "Secreted in brain"...) e ogni gene per specificità nelle cellule del sangue.
Una proteina è marcata "sangue" se:
- è secreta nel sangue e non nel cervello (HPA), oppure è specifica delle cellule del sangue;
- oppure è nella **lista nera** delle proteine plasmatiche classiche: albumina, immunoglobuline, emoglobine,
  fibrinogeno e fattori della coagulazione, apolipoproteine epatiche, complemento, proteine di fase acuta, CA1.

*Domanda da esperto: basta il nome della famiglia per escludere?* No, e qui una lista nera "a nome" sbaglia. Il
complemento C1q, C3 e C4 è **prodotto nel cervello** (microglia e astrociti) ed è centrale nella potatura sinaptica e
nella schizofrenia; la transtiretina (TTR) è prodotta dai plessi corioidei e trasporta la tiroxina nel liquor;
l'apolipoproteina E è la principale apolipoproteina del cervello. Escluderle toglierebbe proteine cerebrali vere.
Per questo ogni criterio è **protetto dall'espressione**: una proteina resta se è espressa nel tessuto cerebrale
(≥ 10 nTPM) o nei neuroni o nella glia (≥ 5 nCPM). Esce solo ciò che il cervello non produce: è arrivato col sangue.
La lista CRAPome (contaminanti degli esperimenti di immunoprecipitazione: cheratine, chaperoni) non c'entra col
sangue e non viene usata.
Con `ESCLUDI_SANGUE = True` le proteine marcate escono da tutto: training, test e predizioni.

In [ ]:
BLOOD_LIST = (r'^(?:ALB|AFP|AFM|IG[HKL][VDJC]\w*|IGH[AGMDE]\d?|IGLL\d|JCHAIN|HB[ABDEGMQZ]\d?|HBA\d|FG[ABG]|F2|F5|F7|F8|F9|'
              r'F10|F11|F12|F13[AB]|KLKB1|SERPINC1|SERPIND1|PLG|KNG1|HRG|HP|HPR|HPX|TF|APO(?:A1|A2|A4|A5|B|C1|C2|C3|C4|F|H|M)|'
              r'AHSG|FETUB|ORM\d|SAA\d|CRP|APCS|AMBP|ITIH\d|C1[RS]|C[2-9]|C8[ABG]|C4[AB]|C4BP[AB]|CFB|CFD|CFH|CFHR\d|CFI|'
              r'MBL2|FCN[23]|CA1|SLC4A1|GYPA|GYPB|EPB42|SPTA1|ALAS2|LTF|MPO|ELANE|PRTN3|DEFA\d|S100A[89])$')
with zipfile.ZipFile(CACHE / 'hpa_proteinatlas.zip') as z:
    atlas = pd.read_csv(z.open(z.namelist()[0]), sep='\t', low_memory=False)
# i nomi delle colonne cambiano tra versioni HPA: le cerco per parola chiave (se manca, la colonna resta vuota)
pick = pd.concat([atlas.filter(regex='(?i)^gene$'), atlas.filter(regex='(?i)secretome location').iloc[:, :1],
                  atlas.filter(regex='(?i)^RNA tissue specificity$').iloc[:, :1], atlas.filter(regex='(?i)^RNA tissue specific nTPM$').iloc[:, :1],
                  pd.DataFrame(index=atlas.index, columns=['_'])], axis=1)
pick.columns = ['gene_hpa', 'secretome', 'tipo', 'tessuti'][:pick.shape[1] - 1] + ['_']
pick = pick.reindex(columns=['gene_hpa', 'secretome', 'tipo', 'tessuti']).astype(str)
pick['gene_hpa'] = pick['gene_hpa'].str.upper()
# proteine del plasma: secrete nel sangue e non nel cervello
to_blood = pick['secretome'].str.contains('(?i)blood') & ~pick['secretome'].str.contains('(?i)brain')
# proteine delle cellule del sangue: espressione arricchita solo in midollo osseo e tessuti linfoidi rispetto a tutti gli
# altri tessuti. La colonna "RNA blood cell specificity" di HPA non serve a questo: misura la specificità tra un tipo di
# cellula immunitaria e l'altro, e marcava come "sangue" proteine come TLR4, GABRP o KLF4
TESSUTI_SANGUE = ['bone marrow', 'lymphoid tissue', 'lymph node', 'spleen', 'thymus', 'tonsil', 'appendix', 'blood']
tess = pick['tessuti'].str.split(';').explode().str.split(':').str[0].str.strip().str.lower()
blood_cell = pick['tipo'].str.contains('(?i)enriched|enhanced') & tess.isin(TESSUTI_SANGUE).groupby(level=0).all()
ctx['secreto_cervello'] = ctx['gene'].isin(pick.loc[pick['secretome'].str.contains('(?i)brain'), 'gene_hpa']).astype(int)
# prodotta nel cervello: tessuto cerebrale (≥ 10 nTPM) oppure neuroni e glia (≥ 5 nCPM)
fatta_nel_cervello = ((ctx['brain_ntpm'] >= 10) | (ctx['neuron_ncpm'] >= NEURON_NCPM) | (ctx['glia_ncpm'] >= NEURON_NCPM)).fillna(False)
# globine, immunoglobuline e membrana dell'eritrocita: nel tessuto cerebrale il loro RNA viene dal sangue che lo irrora,
# per cui la produzione nel cervello si accetta solo dalle cellule singole (neuroni, glia), non dal tessuto intero
INTRINSECHE = r'^(?:HB[ABDEGMQZ]\d?|HBA\d|IG[HKL][VDJC]\w*|IGH[AGMDE]\d?|IGLL\d|JCHAIN|SLC4A1|GYPA|GYPB|EPB42|SPTA1|ALAS2)$'
intrinseca = ctx['gene'].fillna('').str.match(INTRINSECHE)
fatta_nelle_cellule = ((ctx['neuron_ncpm'] >= NEURON_NCPM) | (ctx['glia_ncpm'] >= NEURON_NCPM)).fillna(False)
candidata = (ctx['gene'].isin(pick.loc[to_blood | blood_cell, 'gene_hpa']) | ctx['gene'].fillna('').str.match(BLOOD_LIST))
ctx['is_blood'] = ((candidata & ~fatta_nel_cervello) | (intrinseca & ~fatta_nelle_cellule)).astype(int)
print(f"Proteine marcate 'sangue': {ctx['is_blood'].sum():,} (plasma {ctx['gene'].isin(pick.loc[to_blood, 'gene_hpa']).mul(ctx['is_blood']).sum():,}, "
      f"cellule del sangue {ctx['gene'].isin(pick.loc[blood_cell, 'gene_hpa']).mul(ctx['is_blood']).sum():,}, "
      f"famiglie note {ctx['gene'].fillna('').str.match(BLOOD_LIST).mul(ctx['is_blood']).sum():,}) | candidate tenute perché prodotte nel "
      f"cervello: {(candidata & ~ctx['is_blood'].astype(bool)).sum():,} (es. {', '.join(ctx.loc[candidata & ~ctx['is_blood'].astype(bool), 'gene'].dropna().head(8))})")
print('Controlli:', ctx.loc[ctx['gene'].isin(['HBB', 'HBA1', 'ALB', 'TLR4', 'GABRP', 'KLF4', 'C1QA', 'C3', 'TTR', 'APOE']), ['gene', 'is_blood']]
      .set_index('gene')['is_blood'].to_dict())

# specificità cerebrale (HPA, RNA di consenso HPA + GTEx): "arricchita", "aumentata", "espressa ovunque"...
spec = pd.concat([atlas.filter(regex='(?i)^gene$'), atlas.filter(regex='(?i)^RNA tissue specificity$').iloc[:, :1],
                  atlas.filter(regex='(?i)^RNA tissue specific nTPM$').iloc[:, :1], pd.DataFrame(index=atlas.index, columns=['_'])], axis=1)
spec.columns = ['gene_hpa', 'tipo', 'tessuti'][:spec.shape[1] - 1] + ['_']
spec = spec.reindex(columns=['gene_hpa', 'tipo', 'tessuti']).astype(str)
spec['gene_hpa'] = spec['gene_hpa'].str.upper()
spec['brain_specificity'] = np.where(spec['tessuti'].str.contains('(?i)brain'), spec['tipo'] + ' (brain)',
                                     spec['tipo'].str.replace('nan', 'not available'))
ctx['brain_specificity'] = ctx['gene'].map(spec.drop_duplicates('gene_hpa').set_index('gene_hpa')['brain_specificity'])
print(ctx['brain_specificity'].value_counts().head(6).to_string())

## Percentuali misurate: quanta parte di ogni proteina è libera nel citosol
Due esperimenti umani indipendenti di frazionamento misurano, per ogni proteina, la quota nel pool nucleare,
citosolico e organellare: Hein et al. 2024 (HEK293T, ~8.500 proteine) e Itzhak et al. 2016 (HeLa, ~8.700).
- **frazione citosolica libera**: la più riproducibile tra i due esperimenti (correlazione HEK293T–HeLa ~0,90);
- **frazione nucleare** (correlazione ~0,76): il pool nucleare contiene anche i mitocondri, che sedimentano con i
  nuclei (le proteine mitocondriali risultano "40–60% nucleari"). Il notebook 02 corregge l'artefatto con le
  etichette mitocondriali.
Il notebook 02 le impara come target e le usa per costruire le percentuali per compartimento. Per le proteine
misurate la percentuale finale è la misura stessa.

In [ ]:
noc = pd.read_csv(CACHE / 'hein2024_noc.csv', header=[0, 1], low_memory=False)
hein = pd.DataFrame({'ids': noc['metadata']['Majority protein IDs'].astype(str), 'frac_cyto_hek': noc['sample']['NOC_cytosol'],
                     'frac_nuc_hek': noc['sample']['NOC_nuclear']})
itz = pd.read_excel(CACHE / 'itzhak2016.xlsx', sheet_name='Intensities Static')
itz_tot = itz.filter(regex=r'Intensity MAP\d_(?i:nuclear|cytosolic|organellar)').sum(axis=1)
itz = pd.DataFrame({'ids': itz['Majority protein IDs'].astype(str),
                    'frac_cyto_hela': itz.filter(regex=r'Intensity MAP\d_cytosolic').sum(axis=1) / itz_tot.where(itz_tot > 0),
                    'frac_nuc_hela': itz.filter(regex=r'Intensity MAP\d_(?i:nuclear)').sum(axis=1) / itz_tot.where(itz_tot > 0)})
# collego per accession UniProt (senza suffisso di isoforma): solo accession umane note
up_ids = set(up['uniprot_id'])
frac = []
for tab in [hein, itz]:
    t = tab.assign(uniprot_id=tab['ids'].str.split(';')).explode('uniprot_id')
    t['uniprot_id'] = t['uniprot_id'].str.replace(r'-\d+$', '', regex=True)
    frac.append(t[t['uniprot_id'].isin(up_ids)].drop(columns='ids').drop_duplicates('uniprot_id').set_index('uniprot_id'))
frac = pd.concat(frac, axis=1)
frac['frac_cyto'] = frac[['frac_cyto_hek', 'frac_cyto_hela']].mean(axis=1)
frac['frac_nuc'] = frac[['frac_nuc_hek', 'frac_nuc_hela']].mean(axis=1)
both = frac.dropna()
print(f"Frazione citosolica misurata: {frac['frac_cyto'].notna().sum():,} proteine | in entrambi gli esperimenti: {len(both):,} "
      f"| correlazione HEK293T–HeLa {both['frac_cyto_hek'].corr(both['frac_cyto_hela']):.3f}")
ctx = ctx.join(frac, on='uniprot_id')

## Contesto molecolare: complessi, interattori, abbondanza
La frazione citosolica non dipende solo dalla sequenza: una proteina dentro un grande complesso (ribosoma,
spliceosoma...) sedimenta con esso, e i partner di uno stesso complesso hanno frazioni simili. Test su dati reali
(8.541 proteine): la frazione media delle altre sottounità CORUM predice quella della proteina con Spearman 0,61,
quella degli interattori OpenCell 0,41. Si salvano:
- **CORUM 4.0** (complessi umani curati, per accession);
- **interattoma OpenCell** (IP-MS di 1.310 proteine marcate endogenamente in HEK293T, per simbolo genico);
- **copie per cellula** (OpenCell, HEK293T).
I file vengono dal repository di Hein et al. (stesse versioni usate nell'articolo). Il notebook 02 calcola le medie dei
vicini solo con le etichette di training di ogni fold, per non far trapelare il target.

In [ ]:
cor = pd.read_csv(CACHE / 'corum4.tsv', sep='\t', usecols=['ComplexID', 'ComplexName', 'Organism', 'subunits(UniProt IDs)'])
cor = cor[cor['Organism'] == 'Human'].assign(uniprot_id=cor['subunits(UniProt IDs)'].str.split(';')).explode('uniprot_id')
edges_corum = cor[['ComplexID', 'uniprot_id']].merge(cor[['ComplexID', 'uniprot_id']], on='ComplexID', suffixes=('_a', '_b'))
oc_int = pd.read_csv(CACHE / 'opencell_interactions.csv', usecols=['target_gene_name', 'interactor_gene_name'])
sym2acc = up.drop_duplicates('gene').set_index('gene')['uniprot_id']
edges_oc = pd.DataFrame({'uniprot_id_a': oc_int['target_gene_name'].str.upper().map(sym2acc),
                         'uniprot_id_b': oc_int['interactor_gene_name'].str.upper().map(sym2acc)})
edges_oc = pd.concat([edges_oc, edges_oc.rename(columns={'uniprot_id_a': 'uniprot_id_b', 'uniprot_id_b': 'uniprot_id_a'})])
grafo = pd.concat([edges_corum[['uniprot_id_a', 'uniprot_id_b']].assign(tipo='corum'), edges_oc.assign(tipo='opencell')]).dropna()
grafo = grafo[grafo['uniprot_id_a'] != grafo['uniprot_id_b']].drop_duplicates()
copies = pd.read_csv(CACHE / 'opencell_abundance.csv').drop_duplicates('uniprot_id').set_index('uniprot_id')['hek_protein_copy_number']
ctx['ctx_log_copie_hek'] = np.log10(ctx['uniprot_id'].map(copies))
print(grafo.groupby('tipo')['uniprot_id_a'].agg(['size', 'nunique']).rename(columns={'size': 'archi', 'nunique': 'proteine'}))
ctx = ctx.merge(aux, on='uniprot_id', how='left').merge(ctx_genic, on='uniprot_id', how='left')

## card_protein: il profilo testuale di ogni proteina
La scheda raccoglie, in frasi brevi, quello che un esperto guarda per capire una proteina:
- **identità**: nome, gene, famiglia, lunghezza;
- **funzione**: funzione, attività catalitica (reazione ed EC, senza identificativi), via metabolica;
- **architettura**: domini (nomi dei domini annotati), struttura del complesso e partner, complessi CORUM;
- **regolazione**: modifiche post-traduzionali;
- **ontologie**: processi biologici e funzioni molecolari GO (le prime 25 e 15 voci; **mai** la componente cellulare);
- **cervello**: specificità di tessuto UniProt, espressione massima nelle regioni cerebrali (HPA), nei neuroni, nella
  glia e negli oligodendrociti, gene di sinaptopatia;
- **malattia**: malattie associate (UniProt), con il nome e la descrizione clinica.

Pulizia (le domande da esperto qui sotto spiegano il perché):
- via codici di evidenza, riferimenti PubMed, identificativi MIM, Rhea, ChEBI, GO;
- via ogni parola che nomina un compartimento o un processo di smistamento (nucleo, mitocondri, membrana, secreto,
  sinapsi, vescicola, assone, peptide segnale, transmembrana, traslocazione, import...). Esempio: "Mitochondrial ATP
  synthase subunit" diventa "ATP synthase subunit": resta la funzione, sparisce l'indirizzo;
- via le annotazioni di sequenza (peptide segnale, transit peptide, eliche TM): sono **compiti** che la rete deve
  imparare dalla sequenza, non risposte da leggere.

*Perché non lasciare la localizzazione nella scheda, visto che ai ricercatori interessa?* Perché la scheda è un
ingresso del modello: con la localizzazione dentro, il modello la copierebbe e le prestazioni sembrerebbero ottime
proprio sulle proteine già note, inutili per chi cerca predizioni nuove. Test reale (nomi delle proteine umane di
DeepLoc 1): molti nomi contengono già l'indirizzo, e la versione grezza "migliora" più della versione ripulita.
Le localizzazioni note restano nelle colonne delle fonti, accanto alla scheda, nella tabella finale.

*Perché in inglese?* I modelli di linguaggio biomedici (PubMedBERT) sono addestrati su PubMed: leggono l'inglese.

In [ ]:
LOC_WORDS = (r'(?i)\b(?:nucle\w*|nucleol\w*|mitochondri\w*|cytoplasm\w*|cytosol\w*|membran\w*|endoplasmic|reticul\w*|'
             r'golgi\w*|lysosom\w*|vacuol\w*|peroxisom\w*|secret\w*|extracellular|periplasm\w*|endosom\w*|vesic\w*|'
             r'synap\w*|axon\w*|dendrit\w*|myelin\w*|organell\w*|lumen\w*|centrosom\w*|centriol\w*|chromosom\w*|'
             r'chromatin|cytoskelet\w*|microtubul\w*|plasma|cell surface|apical|basolateral|perinuclear|sarcoplasm\w*|'
             r'sarcolemm\w*|kinetochore|envelope|intermembrane|matri(?:x|ces)|cili(?:um|a|ary)|flagell\w*|spindle|'
             r'signal peptide|signal sequence|transit peptide|transmembrane\w*|anchor\w*|lipoprotein|gpi|translocat\w*|'
             r'import\w*|export\w*|shuttl\w*|targeting|sorting|retention|retriev\w*|compartment\w*|subcellular|'
             r'localiz\w*|located|location|exocyt\w*|endocyt\w*|neurite|growth cone|bouton|postsynaptic|presynaptic|'
             r'nuclear|exosom\w*|lipid droplet|granule|p-body|cajal|speckle|histone|ribosom\w*|outer|inner)\b')
src_txt = up.set_index('uniprot_id')
clean = src_txt[['protein_name', 'function', 'catalytic', 'pathway', 'subunit', 'domain_cc', 'ptm', 'tissue', 'disease',
                 'family', 'go_bp', 'go_mf', 'keywords']].fillna('').replace(
    {r'\{ECO:[^}]*\}': '', r'\(PubMed:[^)]*\)': '', r'\[MIM:\d+\]': '', r'\[GO:\d+\]': '', r'Xref=[^;]*;?': '',
     r'Evidence=[^;]*;?': '', r'PhysiologicalDirection=[^;]*;?': '', r'Note=': '', r'\b[A-Z][A-Z ]{2,}: ': ' '}, regex=True)
domains = src_txt['domain_ft'].fillna('').str.findall(r'/note="([^"]+)"').str.join(', ')
go_bp = clean['go_bp'].str.split(r'\s*;\s*').str[:25].str.join('; ')
go_mf = clean['go_mf'].str.split(r'\s*;\s*').str[:15].str.join('; ')
keywords = clean['keywords'].str.replace(r'(?i)3D-structure|Reference proteome|Alternative splicing|Direct protein sequencing|'
                                         r'Polymorphism|Disease variant|Acetylation', '', regex=True)
complexes = cor.dropna(subset=['uniprot_id']).groupby('uniprot_id')['ComplexName'].agg('; '.join).str.slice(0, 400)
ctx_i = ctx.set_index('uniprot_id')
# i numeri diventano testo con un formato fisso; "nan" (non misurato) diventa "not measured"
num = {c: ctx_i[c].map('{:.1f}'.format).str.replace('nan', 'not measured') for c in ['brain_ntpm', 'neuron_ncpm', 'glia_ncpm', 'oligo_ncpm']}
expr = ('Brain expression (Human Protein Atlas): up to ' + num['brain_ntpm'] + ' nTPM across brain regions; '
        + num['neuron_ncpm'] + ' nCPM in neurons, ' + num['glia_ncpm'] + ' nCPM in glia, ' + num['oligo_ncpm'] + ' nCPM in oligodendrocytes')
parts = [('Protein: ', clean['protein_name']), ('Gene: ', ctx_i['gene']), ('Family: ', clean['family']),
         ('Function: ', clean['function']), ('Catalytic activity: ', clean['catalytic']), ('Pathway: ', clean['pathway']),
         ('Domains: ', domains), ('Domain notes: ', clean['domain_cc']), ('Subunit and partners: ', clean['subunit']),
         ('Protein complexes: ', complexes), ('Post-translational modifications: ', clean['ptm']),
         ('Biological processes: ', go_bp), ('Molecular functions: ', go_mf), ('Keywords: ', keywords),
         ('Tissue specificity: ', clean['tissue']), ('Disease: ', clean['disease'])]
# campi "sicuri" (numeri e liste di malattia scritti da noi): non passano dalla pulizia delle parole di localizzazione
safe = [('', expr), ('Synaptopathy gene (SynaptopathyDB): ', ctx_i['aux_sinaptopatia'].map({1.0: 'yes', 0.0: 'no'})),
        ('Length: ', src_txt['sequence'].str.len().map('{:.0f}'.format) + ' amino acids')]
card_txt, card_safe = pd.Series('', index=ctx_i.index), pd.Series('', index=ctx_i.index)
for label, field in parts:
    f_ = field.reindex(ctx_i.index).fillna('').astype(str).str.strip().str.rstrip('.;')
    card_txt = card_txt + np.where(f_.str.len() > 0, label + f_ + '. ', '')
for label, field in safe:
    f_ = field.reindex(ctx_i.index).fillna('').astype(str).str.strip()
    card_safe = card_safe + np.where(f_.str.len() > 0, label + f_ + '. ', '')
# pulizia: via le parole di localizzazione, poi le preposizioni rimaste orfane ("in the.") e la punteggiatura doppia
card_txt = (card_txt.str.replace(LOC_WORDS, '', regex=True).str.replace(r'\(\s*\)|\[\s*\]', '', regex=True)
            .str.replace(r'(?i)\b(?:in|of|to|into|at|on|from|within|the|a|an)(?:\s+(?:the|a|an))?\s*(?=[.,;:])', '', regex=True)
            .str.replace(r'\s+', ' ', regex=True).str.replace(r'\s+([.,;:])', r'\1', regex=True)
            .str.replace(r'[;,]\s*\.', '.', regex=True).str.replace(r'\.(\s*\.)+', '.', regex=True))
card = (card_txt + card_safe).str.slice(0, 6000)
ctx['card_protein'] = ctx['uniprot_id'].map(card)
# quanto è ricca la scheda: campi di conoscenza compilati (le proteine poco studiate hanno schede povere)
ctx['card_ricchezza'] = ctx['uniprot_id'].map(sum((parts_f.reindex(ctx_i.index).fillna('').astype(str).str.len() > 0).astype(int)
                                                  for parts_f in [clean['function'], clean['family'], go_bp, go_mf, domains,
                                                                  clean['subunit'], clean['ptm'], clean['disease'], clean['catalytic']]))
print(f"card_protein: lunghezza mediana {ctx['card_protein'].str.len().median():.0f} caratteri; "
      f"proteine con funzione descritta {(clean['function'].str.len() > 0).mean():.0%}")
print(ctx.loc[ctx['card_ricchezza'] == ctx['card_ricchezza'].max(), 'card_protein'].iloc[0][:700])

## Merge finale delle proteine
DeepLoc e test HPA restano come sono. Le proteine umane non presenti in DeepLoc entrano solo se almeno una fonte
le osserva positive per un organello o un compartimento neuronale. Duplicati risolti per sequenza identica.

In [ ]:
# entrano anche le proteine con frazione misurata: sono il training delle teste delle percentuali
pos_ids = set(obs.loc[obs['obs'] == 1, 'uniprot_id']) | set(frac.index)
extra = up[up['uniprot_id'].isin(pos_ids) & ~up['sequence'].isin(set(dl['sequence']) | set(hpa_test['sequence']))]
extra = extra[['uniprot_id', 'sequence']].assign(source='human_extra', partition=-1)
proteins = pd.concat([dl[dl['is_human_dl'] | ALTRE_SPECIE][['uniprot_id', 'sequence', 'source', 'partition']],
                      hpa_test.dropna(subset=['uniprot_id'])[['uniprot_id', 'id_test', 'sequence', 'source', 'partition'] + CLASSES[:2] + CLASSES[3:]],
                      extra], ignore_index=True)
proteins['sequence'] = proteins['sequence'].str.upper().str.replace(r'[^A-Z]', '', regex=True)
proteins = proteins[proteins['sequence'].str.len() > 40].drop_duplicates(['source', 'sequence']).reset_index(drop=True)
proteins = proteins.merge(ctx, on='uniprot_id', how='left')
flags = ['is_brain', 'is_neuronal', 'is_oligo', 'is_human', 'is_blood', 'secreto_cervello']
proteins[flags] = proteins[flags].fillna(0).astype(int)
# il perimetro: umane, espresse nel cervello, non del sangue (più le altre specie di DeepLoc solo con ALTRE_SPECIE)
prima = proteins['source'].value_counts()
perimetro = (((proteins['is_human'] == 1) & (proteins['is_brain'] == 1) & ~((proteins['is_blood'] == 1) & ESCLUDI_SANGUE)) |
             ((proteins['is_human'] == 0) & (proteins['source'] == 'deeploc') & ALTRE_SPECIE))
proteins = proteins[perimetro].reset_index(drop=True)
proteins['card_protein'] = proteins['card_protein'].fillna('')
print(pd.DataFrame({'prima del filtro': prima, 'cervello, umane, senza sangue': proteins['source'].value_counts()}))
print(f"Il perimetro seleziona poco: passa il {(ctx['is_brain'].astype(bool) & ~ctx['is_blood'].astype(bool)).mean():.0%} delle proteine "
      f"umane di Swiss-Prot; {ctx['is_neuronal'].mean():.0%} risulta neuronale (≥ {NEURON_NCPM} nCPM nei neuroni)")
proteins['pid'] = np.arange(len(proteins))

# AlphaFold (facoltativo): se il proteoma umano è stato estratto in data/alphafold/, si decomprimono i modelli delle proteine
# del progetto con il nome che usa il notebook 02 (<accession>.cif); senza tar, lo 02 li scarica uno per uno
AFDIR = DATA / 'alphafold'
AFDIR.mkdir(exist_ok=True)
af_gz = sorted(AFDIR.glob('AF-*-F1-model_v*.cif.gz'))
af_acc = [re.sub(r'^AF-|-F1-model_v\d+\.cif\.gz$', '', f.name) for f in af_gz]
da_fare = [a_ in set(proteins['uniprot_id']) and not (AFDIR / f'{a_}.cif').exists() for a_ in af_acc]
for f_gz, a_ in compress(zip(af_gz, af_acc), da_fare):
    (AFDIR / f'{a_}.cif').write_bytes(gzip.decompress(f_gz.read_bytes()))
print(f'Strutture AlphaFold decompresse dal tar: {sum(da_fare):,}')

# le osservazioni puntano alle righe di training/estese; il test HPA tiene le sue etichette nelle colonne CLASSES
train_pid = proteins[proteins['source'] != 'hpa_test'].drop_duplicates('uniprot_id').set_index('uniprot_id')['pid']
obs['pid'] = obs['uniprot_id'].map(train_pid)
obs = obs.dropna(subset=['pid']).astype({'pid': int}).drop_duplicates(['pid', 'fonte', 'classe'])

print(pd.crosstab(proteins['source'], proteins['is_brain']))
print(proteins.groupby('source')[['is_human', 'is_brain', 'is_neuronal', 'is_oligo', 'is_blood']].sum())
print(f"Proteine con frazione citosolica misurata: {proteins['frac_cyto'].notna().sum():,}")
print(obs.pivot_table(index='fonte', columns='classe', values='obs', aggfunc='sum').fillna(0).astype(int).T)
proteins.to_parquet(DATA / 'proteine.parquet', index=False)
obs.to_parquet(DATA / 'osservazioni.parquet', index=False)
proteins[['uniprot_id', 'gene', 'protein_name', 'source', 'card_ricchezza', 'card_protein']].to_excel(DATA / 'card_proteine.xlsx', index=False)
# archi del grafo tra righe di training/estese (pid), nei due versi
# pid_a = la proteina che riceve il contesto (training, estese e test HPA); pid_b = il partner che lo fornisce (mai il test)
test_pid = proteins[proteins['source'] == 'hpa_test'].drop_duplicates('uniprot_id').set_index('uniprot_id')['pid']
grafo = grafo.assign(pid_a=grafo['uniprot_id_a'].map(pd.concat([train_pid, test_pid])), pid_b=grafo['uniprot_id_b'].map(train_pid)).dropna()
grafo[['pid_a', 'pid_b', 'tipo']].astype({'pid_a': int, 'pid_b': int}).to_parquet(DATA / 'grafo.parquet', index=False)

## Varianti: ClinVar (missense, proteine cerebrali) e varianti con localizzazione misurata
P/LP = patogeniche, B/LB = benigne (controllo), VUS. Le "conflicting" sono escluse. La variante si tiene solo se il
residuo WT coincide con la sequenza UniProt (la posizione p. è riferita al trascritto RefSeq).

**Varianti con localizzazione misurata.** Lo screening di Lacoste et al. 2024 (Cell) ha misurato al microscopio la
localizzazione di migliaia di varianti missense; il repository pubblico di Wefers et al. 2026 ne distribuisce 308 con
sequenza normale, mutante ed esito. Si scaricano in automatico e diventano il banco diretto per l'analisi delle
varianti del notebook 03. Lo 02 codifica 4.000 varianti: tutte quelle misurate entrano sempre nel campione, il resto
è estratto a caso tra le patogeniche e le benigne di ClinVar.


In [ ]:
brain = proteins[(proteins['is_brain'] == 1) & proteins['geneid'].notna() & (proteins['source'] != 'hpa_test')]
gene2prot = brain.assign(GeneID=brain['geneid'].str.strip(';').str.split(';')).explode('GeneID')
gene2prot['GeneID'] = pd.to_numeric(gene2prot['GeneID'], errors='coerce')
gene2prot = gene2prot.dropna(subset=['GeneID']).drop_duplicates('GeneID').set_index('GeneID')[['pid', 'gene', 'sequence']]

CLIN = {'Pathogenic': 'P/LP', 'Likely pathogenic': 'P/LP', 'Pathogenic/Likely pathogenic': 'P/LP',
        'Benign': 'B/LB', 'Likely benign': 'B/LB', 'Benign/Likely benign': 'B/LB', 'Uncertain significance': 'VUS'}
AA3 = {'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C', 'Gln': 'Q', 'Glu': 'E', 'Gly': 'G', 'His': 'H',
       'Ile': 'I', 'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P', 'Ser': 'S', 'Thr': 'T', 'Trp': 'W',
       'Tyr': 'Y', 'Val': 'V'}
chunks = []
cols_cv = ['Type', 'Name', 'GeneID', 'ClinicalSignificance', 'Assembly', 'PhenotypeList', 'VariationID']
with gzip.open(CACHE / 'clinvar_variant_summary.txt.gz', 'rt') as fh:
    for ch in pd.read_csv(fh, sep='\t', usecols=cols_cv, chunksize=250_000, low_memory=False):
        ch = ch[(ch['Assembly'] == 'GRCh38') & (ch['Type'] == 'single nucleotide variant') & ch['GeneID'].isin(gene2prot.index)]
        chunks.append(ch.assign(clin=ch['ClinicalSignificance'].map(CLIN)).dropna(subset=['clin']))
cv = pd.concat(chunks, ignore_index=True)
cv[['wt', 'pos', 'mt']] = cv['Name'].str.extract(r'\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)')
cv['wt'], cv['mt'] = cv['wt'].map(AA3), cv['mt'].map(AA3)
cv = cv.dropna(subset=['wt', 'mt']).join(gene2prot, on='GeneID')
cv = cv[cv['wt'] != cv['mt']].astype({'pos': int})
cv = cv[cv['wt'] == [s[p - 1:p] for s, p in zip(cv['sequence'], cv['pos'])]]
cv['brain_phenotype'] = cv['PhenotypeList'].str.contains(
    r'(?i)brain|cerebr|neuro|epilep|seizure|ataxi|dementia|alzheimer|parkinson|intellectual|autism|encephal|'
    r'leukodystroph|microceph|spastic|schizophren', na=False).astype(int)
cv['variante'] = cv['wt'] + cv['pos'].astype(str) + cv['mt']
cv = cv.drop_duplicates(['pid', 'pos', 'mt', 'clin'])[['pid', 'gene', 'VariationID', 'variante', 'wt', 'pos', 'mt', 'clin', 'brain_phenotype']]

print(cv['clin'].value_counts().to_string())
# contesto genico ricalcolato (in FFFFF il download ClinVar era fallito): quota di varianti patogeniche per gene
cv_gene = cv.groupby('pid')['clin'].value_counts().unstack(fill_value=0).reindex(columns=['P/LP', 'B/LB', 'VUS'], fill_value=0)
frac_pat = (cv_gene['P/LP'] + 1) / (cv_gene.sum(axis=1) + 2)

# varianti con localizzazione misurata: lo screening di Lacoste et al. 2024 (Cell) nella versione di Wefers et al. 2026,
# con sequenza normale, mutante ed esito "mislocalizzata" sì/no. Restano le missense delle proteine del perimetro il cui
# residuo normale coincide con la nostra sequenza; quelle che ClinVar non classifica entrano con clin = 'misurata' e
# restano fuori dalla valutazione su ClinVar (P/LP contro B/LB)
lac = pd.read_csv(CACHE / 'lacoste_wefers2026.csv', encoding='utf-8')
lac = lac[lac['mutated'].astype(bool)].copy()
lac[['wt', 'pos', 'mt']] = lac['Variant'].astype(str).str.split().str[-1].str.extract(r'^([A-Z])(\d+)([A-Z])$')
acc2prot = brain.drop_duplicates('uniprot_id').set_index('uniprot_id')[['pid', 'gene', 'sequence']]
lac = lac.dropna(subset=['pos']).join(acc2prot, on='Uniprot').dropna(subset=['pid']).astype({'pos': int, 'pid': int})
lac = lac[(lac['wt'] != lac['mt']) & (lac['wt'] == [s[p - 1:p] for s, p in zip(lac['sequence'], lac['pos'])])]
lac['variante'] = lac['wt'] + lac['pos'].astype(str) + lac['mt']
lab_lac = pd.DataFrame({'gene': lac['gene'], 'variante': lac['variante'], 'mislocalizzata': lac['mislocalized'].astype(int)})
cv = pd.concat([cv, lac.assign(VariationID=np.nan, clin='misurata', brain_phenotype=0)[cv.columns]], ignore_index=True)
cv = cv.drop_duplicates(['pid', 'pos', 'mt'], keep='first').reset_index(drop=True)          # la riga ClinVar ha la precedenza
cv['ctx_frazione_patogeniche_gene'] = cv['pid'].map(frac_pat).fillna(0.5)

# lo 02 codifica N_VARIANTI_02 varianti (MAX_VARIANTS, sezione 0): tutte quelle misurate, il resto estratto a caso tra
# P/LP e B/LB. Senza questa scelta le varianti misurate finirebbero quasi tutte fuori dal campione
N_VARIANTI_02 = 4000
# lo 02 tiene solo le varianti che cadono nella parte codificata (proteine fino a 1.022 residui, altrimenti le due metà
# terminali): lo stesso filtro qui, prima di estrarre, altrimenti nell'esecuzione precedente 789 varianti su 4.000 si perdevano
L_cv = cv['pid'].map(brain.set_index('pid')['sequence'].str.len())
codificabile = ((L_cv <= 1022) | (cv['pos'] <= 511) | (cv['pos'] > L_cv - 511)).to_numpy()
misurata = (cv['pid'].astype(str) + ':' + cv['variante']).isin(lac['pid'].astype(str) + ':' + lac['variante']).to_numpy() & codificabile
pb = cv[~misurata & codificabile & cv['clin'].isin(['P/LP', 'B/LB']).to_numpy()]
cv_02 = pd.concat([cv[misurata], pb.sample(n=min(len(pb), N_VARIANTI_02 - int(misurata.sum())), random_state=42)], ignore_index=True)
cv_02.to_parquet(DATA / 'clinvar_missense.parquet', index=False)
print(f'Varianti per lo 02: {len(cv_02):,} (con localizzazione misurata {int(misurata.sum())}, '
      f'P/LP {(cv_02["clin"] == "P/LP").sum():,}, B/LB {(cv_02["clin"] == "B/LB").sum():,})')

var_files = sorted((EXT / 'varianti').glob('*.csv'))
lab_var = pd.concat([pd.read_csv(f, encoding='utf-8') for f in var_files] + [lab_lac, pd.DataFrame(columns=['gene', 'variante', 'mislocalizzata'])],
                    ignore_index=True)[['gene', 'variante', 'mislocalizzata']].drop_duplicates(['gene', 'variante'])
lab_var['gene'] = lab_var['gene'].astype(str).str.upper()
lab_var.to_parquet(DATA / 'varianti_sperimentali.parquet', index=False)
print(f'Varianti con localizzazione misurata: {len(lab_var):,}')